# Complexity-02b — Le seuil du 3-SAT, mesuré contre la densité

**Navigation** : [Index](README.md) | [Version Python](Complexity-02b-SAT-PhaseTransition-Python.ipynb) | [02 — Vérifier ou trouver](Complexity-02-P-NP-Reduction-Python.ipynb) | [03 — Hiérarchie temporelle](Complexity-03-HierarchieTemporelle-Python.ipynb)

**Objectifs** : construire et contrôler DPLL, mesurer la satisfaisabilité et le coût selon la densité, comparer 2-SAT et 3-SAT, puis observer trois tailles. **Prérequis** : clauses CNF, récursion et C#. **Durée estimée** : 60 minutes.

Les graines sont fixes, mais `System.Random` et `random.Random` ne produisent pas les mêmes instances : les résultats chiffrés de ce carnet ne sont pas ceux du jumeau Python.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Text;
using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
const int N_SWEEP = 50, SEEDS = 12, N_ORACLE = 8, GRAINE_BASE = 20261023;
Console.WriteLine($"Runtime {System.Runtime.InteropServices.RuntimeInformation.FrameworkDescription} ; n={N_SWEEP}, graines={SEEDS}, oracle n={N_ORACLE}");

Runtime .NET 10.0.14 ; n=50, graines=12, oracle n=8


## Instances et solveur

Chaque clause contient $k$ variables distinctes choisies uniformément et des signes indépendants. DPLL simplifie, propage les clauses unitaires puis branche sur la variable de plus petit indice. Un nœud compte un état retiré de la pile ; ce n'est pas une durée.

In [2]:
int[][] Generer(int n, int m, int k, int graine) {
    var rng = new Random(graine); var clauses = new int[m][];
    for (int i=0; i<m; i++) {
        var pool = Enumerable.Range(1,n).ToArray(); var clause = new int[k];
        for (int j=0; j<k; j++) { int t=rng.Next(j,n); (pool[j],pool[t])=(pool[t],pool[j]); clause[j]=rng.NextDouble()<0.5 ? pool[j] : -pool[j]; }
        clauses[i]=clause;
    }
    return clauses;
}
(bool,int) Dpll(int[][] instance) {
    var pile = new Stack<(int[][],Dictionary<int,bool>)>(); pile.Push((instance,new())); int noeuds=0;
    while(pile.Count>0) {
        var (clauses, affectation)=pile.Pop(); noeuds++;
        var a=new Dictionary<int,bool>(affectation); var gardees=new List<int[]>(); bool conflit=false;
        foreach(var clause in clauses) {
            var restants=new List<int>(); bool satisfaite=false;
            foreach(int l in clause) { if(a.TryGetValue(Math.Abs(l),out bool v)) { if(v==(l>0)) { satisfaite=true; break; } } else restants.Add(l); }
            if(!satisfaite) { if(restants.Count==0) { conflit=true; break; } gardees.Add(restants.ToArray()); }
        }
        while(!conflit) {
            var unit=gardees.FirstOrDefault(c=>c.Length==1); if(unit==null) break;
            int l=unit[0]; if(a.TryGetValue(Math.Abs(l),out bool v) && v!=(l>0)) { conflit=true; break; }
            a[Math.Abs(l)]=l>0; var suite=new List<int[]>();
            foreach(var c in gardees) { if(c.Contains(l)) continue; var r=c.Where(x=>x!=-l).ToArray(); if(r.Length==0) { conflit=true; break; } suite.Add(r); }
            gardees=suite;
        }
        if(conflit) continue; if(gardees.Count==0) return (true,noeuds);
        int variable=gardees.SelectMany(c=>c).Min(x=>Math.Abs(x));
        foreach(bool valeur in new[]{true,false}) { var suite=new Dictionary<int,bool>(a); suite[variable]=valeur; pile.Push((gardees.ToArray(),suite)); }
    }
    return (false,noeuds);
}
var exemple=Generer(6,8,3,GRAINE_BASE); Console.WriteLine($"Exemple : {exemple.Length} clauses ; {string.Join(" ; ",exemple.Take(4).Select(c=>string.Join(",",c)))}");

Exemple : 8 clauses ; 1,-4,3 ; -6,-1,-4 ; 3,2,-5 ; 3,6,-4


## Oracle indépendant

L'énumération des $2^8$ affectations contrôle 30 instances, avec les deux verdicts comptés. Ce contrôle borné n'est pas une preuve générale.

In [3]:
bool Oracle(int[][] clauses,int n) {
    for(int bits=0;bits<(1<<n);bits++) if(clauses.All(c=>c.Any(l=>(((bits>>(Math.Abs(l)-1))&1)==1)==(l>0)))) return true;
    return false;
}
int desaccords=0, satOracle=0;
for(int i=0;i<30;i++) { var x=Generer(N_ORACLE,(int)Math.Round(4.26*N_ORACLE),3,GRAINE_BASE+1000+i); bool o=Oracle(x,N_ORACLE); if(o) satOracle++; if(Dpll(x).Item1!=o) desaccords++; }
Console.WriteLine($"Oracle : 30 instances, {satOracle} satisfaisables, {30-satOracle} insatisfaisables, {desaccords} désaccords");

Oracle : 30 instances, 24 satisfaisables, 6 insatisfaisables, 0 désaccords


### Exercice 1 — vérifier une affectation

Écrire un vérificateur et compter les littéraux lus. La validation utilise plusieurs clauses et affectations ; compléter le stub pour l'activer. Ce sera aussi l'instrument de comparaison certificat/recherche de la fin du carnet.

In [4]:
(bool,int)? Verifier(int[][] clauses, bool[] valeurs) { // TODO : lire chaque clause jusqu'à son premier littéral vrai
    return null;
}
var testCert=Verifier(new[]{new[]{1,-2},new[]{2}},new[]{true,true});
if(testCert is null) Console.WriteLine("Exercice 1 à compléter : vérificateur et nombre de lectures.");
else { var autre=Verifier(new[]{new[]{1},new[]{-1}},new[]{true}); var longue=Verifier(new[]{new[]{-1,2,3},new[]{1}},new[]{true,false,true}); var fausse=Verifier(new[]{new[]{1,2},new[]{-1,-2}},new[]{true,true}); if(testCert.Value.Item1!=true || testCert.Value.Item2!=2 || autre is null || autre.Value.Item1 || autre.Value.Item2!=2 || longue is null || !longue.Value.Item1 || longue.Value.Item2!=4 || fausse is null || fausse.Value.Item1 || fausse.Value.Item2!=3) throw new Exception("Vérificateur incorrect"); Console.WriteLine("Fixtures du certificat validées."); }

Exercice 1 à compléter : vérificateur et nombre de lectures.


## Balayage 3-SAT

À $n=50$, huit densités entre 2 et 6 ; douze graines par point. La fraction satisfaisable, la médiane et le maximum décrivent des aspects différents. Les points ne sont pas des échantillons appariés entre langages.

In [5]:
record Mesure(double Densite,int M,double PSat,double Mediane,int Maximum);
List<Mesure> Balayer(int n,int k,double[] densites,int nbGraines) {
    var resultats=new List<Mesure>(); foreach(double densite in densites) {
        int m=(int)Math.Round(densite*n), sat=0; var couts=new List<int>();
        for(int graine=0;graine<nbGraines;graine++) { var (ok,compte)=Dpll(Generer(n,m,k,graine)); if(ok)sat++; couts.Add(compte); }
        couts.Sort(); double med=(couts[(nbGraines-1)/2]+couts[nbGraines/2])/2.0;
        resultats.Add(new(densite,m,(double)sat/nbGraines,med,couts.Max()));
    } return resultats;
}
void Afficher(IEnumerable<Mesure> rs) { Console.WriteLine("m/n     m   P(sat)    médiane  maximum"); foreach(var r in rs) Console.WriteLine($"{r.Densite,4:F2} {r.M,5} {r.PSat,8:F2} {r.Mediane,10:F1} {r.Maximum,8}"); }
double[] d3={2,3,3.5,4,4.26,4.5,5,6}; var chrono=Stopwatch.StartNew(); var r3=Balayer(N_SWEEP,3,d3,SEEDS); chrono.Stop(); Afficher(r3); Console.WriteLine($"Durée : {chrono.Elapsed.TotalSeconds:F2} s");

m/n     m   P(sat)    médiane  maximum


2.00   100     1.00       23.0       27


3.00   150     1.00       41.0      640


3.50   175     1.00      124.5      946


4.00   200     0.92      302.5     3191


4.26   213     0.75      413.5     1587


4.50   225     0.50      556.0     2400


5.00   250     0.17      416.0     1574


6.00   300     0.00      198.0     1281


Durée : 4.37 s


Les tableaux sont des mesures finies : un maximum n'est que le plus grand coût parmi douze essais. La position du pic peut varier avec la graine, la statistique et la règle de branchement.

Ici la fraction passe de 1,00 à 3,50 à 0,17 à 5,00 ; le maximum atteint 3 191 nœuds à 4,00 et la médiane 556 à 4,50. La dispersion entre essais rend ces deux pics différents.

In [6]:
record SvgHtml(string Content);
Formatter.Register<SvgHtml>((v,w)=>w.Write(v.Content),"text/html");
string Graphe(string titre,string axeY, IEnumerable<(string nom,string couleur,IEnumerable<(double x,double y)> points)> series,double xmin,double xmax,double ymin,double ymax) {
    string F(double x)=>x.ToString("0.##",CultureInfo.InvariantCulture); double X(double x)=>75+(x-xmin)/(xmax-xmin)*600; double Y(double y)=>300-(y-ymin)/(ymax-ymin)*240;
    var b=new StringBuilder(); b.Append("<svg xmlns='http://www.w3.org/2000/svg' width='760' height='380' viewBox='0 0 760 380'><rect width='760' height='380' fill='white'/>");
    b.Append($"<text x='75' y='22' font-size='15'>{System.Security.SecurityElement.Escape(titre)}</text><path d='M75 60 V300 H675' stroke='black' fill='none'/>");
    for(int i=0;i<=4;i++) { double x=xmin+(xmax-xmin)*i/4, y=ymin+(ymax-ymin)*i/4; b.Append($"<text x='{F(X(x)-12)}' y='320' font-size='11'>{F(x)}</text><text x='20' y='{F(Y(y)+4)}' font-size='11'>{F(y)}</text><path d='M75 {F(Y(y))} H675' stroke='#ddd'/>"); }
    int leg=0; foreach(var (nom,couleur,pts) in series) { var p=pts.ToArray(); b.Append($"<polyline fill='none' stroke='{couleur}' stroke-width='2' points='{string.Join(" ",p.Select(t=>$"{F(X(t.x))},{F(Y(t.y))}"))}'/>"); foreach(var t in p) b.Append($"<circle cx='{F(X(t.x))}' cy='{F(Y(t.y))}' r='3' fill='{couleur}'/>"); b.Append($"<text x='{80+leg*210}' y='355' fill='{couleur}' font-size='12'>{System.Security.SecurityElement.Escape(nom)}</text>"); leg++; }
    b.Append($"<text x='300' y='338' font-size='12'>densité m/n</text><text x='10' y='47' font-size='11'>{System.Security.SecurityElement.Escape(axeY)}</text></svg>"); return b.ToString();
}
var satSvg=Graphe("3-SAT : fraction satisfaisable","fraction",new[]{("P(sat)","#225ea8",r3.Select(r=>(r.Densite,r.PSat)))},2,6,0,1);
display(new SvgHtml(satSvg)); Console.WriteLine("Figure : fraction satisfaisable par densité.");

3-SAT : fraction satisfaisable 2 0 3 0.25 4 0.5 5 0.75 6 1 P(sat) densité m/n fraction

Figure : fraction satisfaisable par densité.


La première figure montre la fraction satisfaisable ; le graphique suivant montre les coûts en échelle logarithmique décimale (les valeurs brutes figurent dans le tableau). Une coïncidence des pics n'est pas imposée par construction.

In [7]:
var coutSvg=Graphe("3-SAT : coût de DPLL","log10 nœuds",new[]{("médiane","#238443",r3.Select(r=>(r.Densite,Math.Log10(r.Mediane)))),("maximum","#b45309",r3.Select(r=>(r.Densite,Math.Log10(r.Maximum))))},2,6,0,Math.Ceiling(Math.Log10(r3.Max(r=>r.Maximum))));
display(new SvgHtml(coutSvg));
Console.WriteLine($"Pic médiane à {r3.OrderByDescending(r=>r.Mediane).First().Densite:F2} ; pic maximum à {r3.OrderByDescending(r=>r.Maximum).First().Densite:F2}.");

3-SAT : coût de DPLL 2 0 3 1 4 2 5 3 6 4 médiane maximum densité m/n log10 nœuds

Pic médiane à 4.50 ; pic maximum à 4.00.


### Exercice 2 — 90e centile

Implémenter le 90e centile de `statistics.quantiles(noeuds, n=10)[8]` : méthode exclusive, rang $0,9(n+1)$ à partir de 1, avec extrapolation linéaire aux bords. Les fixtures non triées couvrent deux tailles ; une fois la fonction complétée, les coûts des vrais balayages sont recalculés et le tableau montre son pic.

In [8]:
double? Centile90(int[] valeurs) { // TODO : trier une copie ; méthode exclusive, rang 0,9(n+1)
    return null;
}
var p90=Centile90(new[]{12,1,11,2,10,3,9,4,8,5,7,6});
if(p90 is null) Console.WriteLine("Exercice 2 à compléter : calculer et comparer le centile 90.");
else {
    var second=Centile90(new[]{10,1,9,2,8,3,7,4,6,5});
    if(!double.IsFinite(p90.Value) || Math.Abs(p90.Value-11.7)>1e-9 || second is null || !double.IsFinite(second.Value) || Math.Abs(second.Value-9.9)>1e-9) throw new Exception("Centile incorrect");
    var table=new List<(double densite,double p90)>();
    foreach(double densite in d3) {
        int m=(int)Math.Round(densite*N_SWEEP); var couts=Enumerable.Range(0,SEEDS).Select(g=>Dpll(Generer(N_SWEEP,m,3,g)).Item2).ToArray();
        var p=Centile90(couts); if(p is null || !double.IsFinite(p.Value)) throw new Exception("Centile manquant ou non fini");
        var tries=couts.OrderBy(x=>x).ToArray(); double attendu=tries[10]+0.7*(tries[11]-tries[10]);
        if(Math.Abs(p.Value-attendu)>1e-9) throw new Exception("Centile incorrect sur les coûts mesurés");
        table.Add((densite,p.Value)); Console.WriteLine($"m/n={densite:F2} : médiane {r3.First(r=>r.Densite==densite).Mediane:F1}, p90 {p.Value:F1}, maximum {couts.Max()}");
    }
    Console.WriteLine($"Pic p90 : m/n={table.OrderByDescending(x=>x.p90).First().densite:F2}");
}

Exercice 2 à compléter : calculer et comparer le centile 90.


## Contrôle 2-SAT

Même générateur, solveur et $n$ ; $k=2$. La comparaison des pics porte sur des densités différentes : elle ne démontre pas une classe de complexité. Le 2-SAT possède un algorithme polynomial ; le 3-SAT est NP-complet. Ce sont des faits théoriques distincts des nœuds mesurés ici.

In [9]:
double[] d2={0.5,0.8,1,1.2,2,4.26}; chrono.Restart(); var r2=Balayer(N_SWEEP,2,d2,SEEDS); chrono.Stop(); Afficher(r2); Console.WriteLine($"Durée : {chrono.Elapsed.TotalSeconds:F2} s");
Console.WriteLine($"Pics observés : 2-SAT {r2.Max(r=>r.Maximum)} nœuds ; 3-SAT {r3.Max(r=>r.Maximum)} nœuds.");

m/n     m   P(sat)    médiane  maximum


0.50    25     1.00       14.0       17


0.80    40     1.00       13.5       19


1.00    50     1.00       12.5       19


1.20    60     0.92       12.0       15


2.00   100     0.00        3.0       11


4.26   213     0.00        3.0        5


Durée : 0.03 s


Pics observés : 2-SAT 19 nœuds ; 3-SAT 3191 nœuds.


Les maxima et fractions du tableau précèdent la comparaison graphique. La propagation unitaire est identique dans les deux expériences ; la forme des clauses change et les densités critiques ne coïncident pas.

Le maximum 2-SAT vaut 19 nœuds dans ce balayage contre 3 191 pour 3-SAT ; à densité commune 4,26 les maxima sont 5 et 1 587. Ces comparaisons ne prouvent pas un rapport universel.

La propagation des clauses unitaires éclaire le mécanisme observé : elle force des variables et peut révéler une contradiction sans nouveau branchement. Les 2-CNF denses en produisent souvent rapidement, alors que les clauses à trois littéraux peuvent laisser davantage de choix avant de devenir unitaires. Les deux pics proviennent cependant de densités différentes : ils décrivent cet échantillon et cette heuristique, pas une mesure des classes de complexité. À densité commune 4,26, les maxima de 5 et 1 587 isolent la comparaison à nombre de clauses égal ; les distributions et la difficulté typique restent distinctes du maximum.

In [10]:
var comparaison=Graphe("Comparaison des maxima","log10 nœuds",new[]{("3-SAT","#b91c1c",r3.Select(r=>(r.Densite,Math.Log10(r.Maximum)))),("2-SAT","#225ea8",r2.Select(r=>(r.Densite,Math.Log10(r.Maximum))))},0.5,6,0,Math.Ceiling(Math.Log10(Math.Max(r2.Max(r=>r.Maximum),r3.Max(r=>r.Maximum)))));
display(new SvgHtml(comparaison)); Console.WriteLine("Maxima comparés sur le même axe de densité.");

Comparaison des maxima 0.5 0 1.88 1 3.25 2 4.63 3 6 4 3-SAT 2-SAT densité m/n log10 nœuds

Maxima comparés sur le même axe de densité.


### Exercice 3 — recentrer les seuils

Balayer 2-SAT à 0,7 / 0,9 / 1 / 1,1 / 1,3 puis comparer ses abscisses $m/n-1$ avec celles du 3-SAT $m/n-4,26$. Les deux tableaux ont des tailles différentes.

In [11]:
#nullable enable
double[]? Recentrer(double[] densites,double seuil) { // TODO : retourner un nouveau tableau
    return null;
}
var centre=Recentrer(new[]{0.7,1.0,1.3},1.0);
if(centre is null) Console.WriteLine("Exercice 3 à compléter : recentrer les deux balayages et tracer les coûts.");
else {
    var autre=Recentrer(new[]{3.5,4.26},4.26);
    if(centre.Length!=3 || centre.Any(x=>!double.IsFinite(x)) || Math.Abs(centre[0]+0.3)>1e-9 || Math.Abs(centre[1])>1e-9 || Math.Abs(centre[2]-0.3)>1e-9 || autre is null || autre.Length!=2 || autre.Any(x=>!double.IsFinite(x)) || Math.Abs(autre[0]+0.76)>1e-9 || Math.Abs(autre[1])>1e-9) throw new Exception("Recentrage incorrect");
    double[] densitesEx={0.7,0.9,1.0,1.1,1.3}; var resEx=Balayer(N_SWEEP,2,densitesEx,SEEDS);
    var x2=Recentrer(densitesEx,1.0); var x3=Recentrer(d3,4.26);
    if(x2 is null || x3 is null || x2.Length!=densitesEx.Length || x3.Length!=d3.Length || x2.Any(x=>!double.IsFinite(x)) || x3.Any(x=>!double.IsFinite(x)) || x2.Where((x,i)=>Math.Abs(x-(densitesEx[i]-1))>1e-9).Any() || x3.Where((x,i)=>Math.Abs(x-(d3[i]-4.26))>1e-9).Any()) throw new Exception("Abscisses recentrées incorrectes");
    var svg=Graphe("Coûts recentrés sur les seuils", "log10 nœuds",new[]{("2-SAT","#225ea8",resEx.Select((r,i)=>(x2[i],Math.Log10(r.Maximum)))),("3-SAT","#b91c1c",r3.Select((r,i)=>(x3[i],Math.Log10(r.Maximum))))},-2.26,1.74,0,Math.Ceiling(Math.Log10(Math.Max(resEx.Max(r=>r.Maximum),r3.Max(r=>r.Maximum)))));
    display(new SvgHtml(svg.Replace("densité m/n","écart au seuil"))); Console.WriteLine("Balayage 2-SAT recentré : 5 densités, 12 graines chacune.");
}

Exercice 3 à compléter : recentrer les deux balayages et tracer les coûts.


## La bande change-t-elle avec la taille ?

On reprend les cinq densités communes pour $n=20,30,40$, douze instances chacune. Une proportion proche de 0,5 possède une erreur type d'environ 0,14 à cette taille d'échantillon : une tendance apparente n'établit pas une loi asymptotique.

In [12]:
double[] da={3.5,4,4.26,4.5,5}; var tailles=new[]{20,30,40};
foreach(int n in tailles) { var r=Balayer(n,3,da,SEEDS); Console.WriteLine($"n={n} fractions : {string.Join(" ",r.Select(x=>x.PSat.ToString("F2")))}"); Console.WriteLine($"n={n} médianes : {string.Join(" ",r.Select(x=>x.Mediane.ToString("F1")))}"); }
Console.WriteLine("Colonnes m/n : 3.50 4.00 4.26 4.50 5.00");

n=20 fractions : 1.00 0.83 0.67 0.58 0.42


n=20 médianes : 9.0 22.5 29.5 36.0 27.0


n=30 fractions : 1.00 0.92 0.75 0.58 0.00


n=30 médianes : 22.5 35.5 63.5 100.0 105.0


n=40 fractions : 0.92 0.92 0.83 0.25 0.08


n=40 médianes : 47.0 193.0 223.5 316.0 258.0


Colonnes m/n : 3.50 4.00 4.26 4.50 5.00


## Portée et exercices

À 4,26 les proportions sont 0,67, 0,75 et 0,83 pour les tailles 20, 30 et 40 ; elles ne montrent pas ici de resserrement monotone. Les mêmes graines sont réutilisées et l'indépendance entre tailles n'est pas établie.

La prédiction 4,267 pour le seuil 3-SAT est **citée**, non démontrée par ces mesures. Le théorème de seuil net de Friedgut ne donne pas cette valeur. Les coûts concernent cet algorithme et son choix de branchement ; douze essais et des tailles finies ne localisent pas une troisième décimale. Les références classiques sont Cheeseman, Kanefsky et Taylor (IJCAI 1991), et Mitchell, Selman et Levesque (AAAI 1992).

Dans cette exécution, l'oracle trouve 24 SAT et 6 UNSAT sans désaccord ; la médiane 3-SAT culmine à 556 nœuds pour 4,50 et le maximum à 3 191 pour 4,00. Le maximum 2-SAT est 19 sur son propre balayage. Ces nombres ne sont ni des constantes universelles ni une preuve des classes de complexité.

Les trois exercices répartis demandent : vérifier un certificat et comparer lectures et nœuds ; calculer le 90e centile et localiser son pic ; recentrer et tracer les deux profils. Pour la comparaison du certificat, adapter DPLL pour retourner son affectation, puis vérifier cette même affectation : les unités « lectures de littéraux » et « nœuds » restent distinctes.

## À retenir

L'oracle confronte DPLL à l'exhaustif sur 30 petites instances ; les balayages montrent des fractions satisfaisables et des coûts pour deux familles et plusieurs tailles. La taille de l'échantillon et l'heuristique limitent la portée des conclusions.

## Vérifiez votre compréhension

1. Pourquoi la valeur 4,267 reste-t-elle citée ? 2. Pourquoi un pic 2-SAT ne démontre-t-il pas une classe de complexité ? 3. Pourquoi médiane et maximum peuvent-ils culminer ailleurs ?

<details><summary>Réponses</summary>

1. Les tailles et l'échantillon ne résolvent pas la troisième décimale asymptotique. 2. Le compteur décrit seulement ce solveur sur ces instances ; la classification est un résultat théorique. 3. Le maximum dépend d'une seule instance, la médiane de l'ensemble des coûts.
</details>

## Pour aller plus loin

- [XOR-SAT : transition sans le mur](Complexity-02c-XOR-SAT-Phase-Transition-Python.ipynb).
- [P, NP et réduction](Complexity-02-P-NP-Reduction-Python.ipynb).

**Navigation** : [Index](README.md) | [Version Python](Complexity-02b-SAT-PhaseTransition-Python.ipynb)